# Power-law (sub-exponential) growth — and why assuming exponential overestimates

v0.40 added von Bertalanffy (surface-limited, *bounded*). This adds the other sub-exponential law and
the empirically best-supported one: the **power-law** `dV/dt = a·V^p` (p < 1). Its specific growth rate
`a·V^{p−1}` falls with size, so growth is slower than any exponential — but *unbounded* (no carrying
capacity). Benzekry et al. (2014) found it the best-fitting unperturbed law across many tumor datasets.

The consequence is a silent model-selection risk at the *growth* layer: the exponential assumption
(`p = 1`) **overestimates** extrapolated tumor burden — the growth-law analog of the v0.36 exposure-
response dose-extrapolation axis. *Unperturbed growth law; illustrative values; unverified.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.export.registry import get_kernel, kernel_values
from scipy.integrate import solve_ivp

ds = onkos.load()
spec = get_kernel(ds['growth_laws.power_law'])
v = kernel_values(ds['growth_laws.power_law']); v['V0'] = 10.0
a, p = v['a'], v['p']

## Closed form, round-trip validated

Separable: `V^{1−p}` is linear in t, giving `V(t) = (V0^{1−p} + a(1−p)t)^{1/(1−p)}`. Round-trip
validated (analytic↔SciPy integration, SBML MathML per state, NONMEM/rxode2/Pumas params) like every
ODE kernel — and like von Bertalanffy it uses a fractional-power rate law.

In [ ]:
t = np.linspace(0, 104, 209)
ana = spec.analytic(t, v)
ode = solve_ivp(lambda tt,y: spec.rhs(tt,y,v), (0,104), [10.0], t_eval=t, rtol=1e-10, atol=1e-12).y[0]
print(f'a={a}  p={p}   max rel err analytic vs ODE = {np.max(np.abs(ana-ode)/ode):.1e}')
print(f'V(0)={ana[0]:.0f}  V(52)={np.interp(52,t,ana):.0f}  V(104)={ana[-1]:.0f}  (unbounded, sub-exponential)')

## Assuming exponential overestimates extrapolated burden

Match an exponential to the power-law's *instantaneous* growth rate at baseline
(`kg = a·V0^{p−1}`), so they share the same early trajectory. They diverge enormously on
extrapolation — the exponential assumption inflates the predicted burden.

In [ ]:
kg = a * v['V0']**(p - 1)
power = spec.analytic(t, v)
expo = v['V0'] * np.exp(kg * t)
for wk in (26, 52, 78, 104):
    pw, ex = np.interp(wk,t,power), np.interp(wk,t,expo)
    print(f'week {wk:>3}:  power-law {pw:>8.0f}   exponential {ex:>12.0f}   overestimate {ex/pw:>6.1f}x')

## The takeaway

The growth-law family now spans the full range: unbounded exponential, bounded logistic/Gompertz/von
Bertalanffy, and unbounded-but-sub-exponential power-law (the empirically favored one). The growth-law
assumption is invisible early but dominates the extrapolated burden — assuming exponential, the field's
convenient default, systematically overestimates. A growth law describes tumor dynamics, never a patient
outcome; tier B, illustrative, unverified.